# Dataset Frequency Analysis

Statistical overview of sensor topic frequencies across all episodes in a dataset.  
Use this to verify recording consistency and identify problematic episodes before training.

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
from mcap.reader import make_reader

from example_policies.data_ops.config.rosbag_topics import RosTopicEnum
from example_policies.data_ops.utils.conversion_utils import get_selected_episodes

# extract_sensor_timestamp handles all message types including CompressedVideo
# (registers custom ROS types internally — no separate side-effect import needed)
from example_policies.data_ops.pipeline.timestamp_utils import extract_sensor_timestamp

# ── Seaborn-inspired style ─────────────────────────────────────────────
plt.rcParams.update({
    # Figure
    "figure.facecolor": "#f0f0f0",
    "figure.dpi": 130,
    "figure.titlesize": 15,
    "figure.titleweight": "bold",
    # Axes
    "axes.facecolor": "#eaeaf2",
    "axes.edgecolor": "white",
    "axes.linewidth": 0,
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.labelsize": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "medium",
    "axes.titlepad": 10,
    # Grid
    "grid.color": "white",
    "grid.linewidth": 1.2,
    "grid.linestyle": "-",
    # Spines
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": False,
    # Ticks
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "xtick.color": "#555555",
    "ytick.color": "#555555",
    # Font
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial", "sans-serif"],
    "font.size": 11,
    "text.color": "#333333",
    "axes.labelcolor": "#333333",
    # Legend
    "legend.frameon": False,
    "legend.fontsize": 9,
    # Savefig
    "savefig.facecolor": "#f0f0f0",
    "savefig.edgecolor": "#f0f0f0",
})

# Palette — seaborn "muted" (desaturated, accessible)
PALETTE = ["#4878d0", "#ee854a", "#6acc64", "#d65f5f", "#956cb4",
           "#8c613c", "#dc7ec0", "#797979", "#d5bb67", "#82c6e2"]

# Slightly darker colours for spike-timeline (5a) — one shade richer than PALETTE
_CLR_OK = "#3a65c2"      # deeper blue (PALETTE[0] = #4878d0)
_CLR_VIOL = "#c44545"    # deeper red  (PALETTE[3] = #d65f5f)
_CLR_THRESH = "#c44545"  # threshold line (same deeper red)

---
## 1. Configuration

In [ ]:
    ("TCP L",       [RosTopicEnum.ACTUAL_TCP_LEFT.value, "/left/franka_robot_state_broadcaster/current_pose", "/panda_left/tcp"]),
    ("TCP R",      [RosTopicEnum.ACTUAL_TCP_RIGHT.value, "/right/franka_robot_state_broadcaster/current_pose", "/panda_right/tcp"]),

---
## 2. Collect per-episode frequency data

Iterates over every episode, extracts sensor timestamps (falling back to log-time), and computes per-topic statistics.

In [ ]:
# ── Extract dataset version(s) from all episodes ─────────────────────
dataset_versions: set[str] = set()
for ep_path in episode_paths:
    v = extract_dataset_version(ep_path)
    if v:
        dataset_versions.add(v)
dataset_version_str = ", ".join(sorted(dataset_versions)) if dataset_versions else None
print(f"Version:   {dataset_version_str or '(not found)'}")

---
## 3. Summary table

Mean, std, min, max frequency per topic across all episodes.

In [ ]:
summary = (
    df[df["avg_hz"] > 0]
    .groupby("topic")["avg_hz"]
    .agg(["mean", "std", "min", "max", "count"])
    .rename(columns={"mean": "mean_hz", "std": "std_hz",
                      "min": "min_hz", "max": "max_hz",
                      "count": "episodes"})
    .sort_values("mean_hz", ascending=False)
)
summary.style.format("{:.1f}").set_caption("Per-topic average frequency (Hz) across episodes")

---
## 4. Average frequency per topic across episodes

Each dot is one episode. The box shows the interquartile range. The dashed line marks the target FPS.

In [ ]:
# Use declaration order from TOPICS config (reversed so first topic is at top)
topic_order = [label for label, _ in reversed(TOPICS)]

OUTPUT_DIR = pathlib.Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(9, max(3, len(topic_order) * 0.5)))

# Box plot (no fliers - we show them as strip)
bp = ax.boxplot(
    [df[(df["topic"] == t) & (df["avg_hz"] > 0)]["avg_hz"].values for t in topic_order],
    vert=False, patch_artist=True, showfliers=False,
    widths=0.55,
    boxprops=dict(facecolor=PALETTE[0], edgecolor="white", linewidth=0.8, alpha=0.35),
    medianprops=dict(color=PALETTE[0], linewidth=2),
    whiskerprops=dict(color=PALETTE[0], linewidth=1.2, alpha=0.6),
    capprops=dict(color=PALETTE[0], linewidth=1.2, alpha=0.6),
)
ax.set_yticklabels(topic_order)

# Strip plot (individual episodes)
for i, t in enumerate(topic_order):
    vals = df[(df["topic"] == t) & (df["avg_hz"] > 0)]["avg_hz"].values
    jitter = np.random.default_rng(42).uniform(-0.15, 0.15, size=len(vals))
    ax.scatter(vals, np.full_like(vals, i + 1) + jitter,
               s=14, alpha=0.55, color=PALETTE[0], edgecolors="white",
               linewidths=0.3, zorder=3)

# Reference line
ax.axvline(TARGET_FPS, ls="--", lw=1.5, color=PALETTE[3], alpha=0.8,
           label=f"Target {TARGET_FPS} Hz")
ax.set_xlabel("Average frequency (Hz)")
ax.legend(loc="lower right")
ax.set_title("Per-episode average frequency by topic")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"avg_frequency_boxplot_{DATASET_LABEL}.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 4b. Per-message instantaneous frequency (violin)

Each violin shows the distribution of **all individual message-to-message frequencies** across all episodes (not episode averages). This reveals jitter, bursts, and timing outliers within episodes.

In [ ]:
# Build per-message instantaneous frequencies from all_intervals (ms → Hz)
# Use declaration order from TOPICS (same as plot 1)
FREQ_CLIP_HZ = 1200  # discard outliers above this

inst_order = []
inst_data = []
for label in topic_order:
    if label in all_intervals and len(all_intervals[label]) > 0:
        intervals_ms = np.array(all_intervals[label])
        freqs = 1000.0 / intervals_ms  # ms → Hz
        freqs = freqs[freqs <= FREQ_CLIP_HZ]
        if len(freqs) > 0:
            inst_order.append(label)
            inst_data.append(freqs)

fig, ax = plt.subplots(figsize=(9, max(3, len(inst_order) * 0.55)))

if inst_data:
    parts = ax.violinplot(
        inst_data, positions=range(len(inst_order)),
        vert=False, showmedians=True, showextrema=False,
    )
    for pc in parts["bodies"]:
        pc.set_facecolor(PALETTE[0])
        pc.set_edgecolor("#aaaaaa")
        pc.set_linewidth(0.8)
        pc.set_alpha(0.65)
    parts["cmedians"].set_color("#333333")
    parts["cmedians"].set_linewidth(1.8)

# Target FPS — small line snippet pinned to the x-axis edge
from matplotlib.transforms import blended_transform_factory
_trans = blended_transform_factory(ax.transData, ax.transAxes)
ax.plot(TARGET_FPS, 0, marker="|", markersize=8, markeredgewidth=1.5,
        color=PALETTE[3], clip_on=False, zorder=5, transform=_trans)
ax.annotate(f"{TARGET_FPS} Hz", xy=(TARGET_FPS, 0), xycoords=_trans,
            xytext=(3, -8), textcoords="offset points",
            fontsize=7, color=PALETTE[3], fontweight="medium",
            clip_on=False, va="top", ha="left")

ax.set_yticks(range(len(inst_order)))
ax.set_yticklabels(inst_order)
ax.set_xlabel("Frequency (Hz)")
ax.set_title(f"Per-message frequency distribution — {DATASET_TITLE}")

fig.tight_layout()

fig.savefig(OUTPUT_DIR / f"instantaneous_frequency_{DATASET_LABEL}.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 5. Tolerance violation debugging

The next three plots help diagnose **when** and **where** inter-message intervals exceed the sync tolerance.

- **5a  Interval spike timeline** — per-episode drill-down showing interval vs. time, with violations highlighted  
- **5b  Violation position histogram** — aggregate: _where_ in an episode do violations occur (start / middle / end)?  
- **5c  Episode × topic violation heatmap** — which episode/topic pairs are the worst offenders?

In [ ]:
# ── 5a. Interval spike timeline — per-episode drill-down ──────────────
# Shows inter-message interval (ms) vs. elapsed time for selected episodes.
# Points above the tolerance line are coloured red.
# Green vertical lines mark AV1 keyframe positions (video topics only).

topic_order_fwd = [label for label, _ in TOPICS]

# Pick episodes to drill into
if DRILL_EPISODES is not None:
    drill_eps = DRILL_EPISODES
else:
    # All episodes that have ≥1 violation on any topic
    drill_eps = []
    for ep_idx, ep_ts in episode_timestamps.items():
        for label in topic_order_fwd:
            ts = ep_ts.get(label)
            if ts is None or len(ts) < 2:
                continue
            intervals_ms = np.diff(ts) * 1000
            if np.any(intervals_ms > actual_tolerance_ms):
                drill_eps.append(ep_idx)
                break
    drill_eps.sort()

print(f"Drill-down episodes: {len(drill_eps)} with violations  (tolerance = {actual_tolerance_ms:.1f} ms)")

for ep_idx in drill_eps:
    ep_ts = episode_timestamps.get(ep_idx)
    if ep_ts is None:
        continue
    ep_kf = episode_keyframes.get(ep_idx, {})

    # Count how many topics actually have data
    active_topics = [lbl for lbl in topic_order_fwd
                     if lbl in ep_ts and len(ep_ts[lbl]) >= 2]
    if not active_topics:
        continue

    # Compute a common t0 across all topics for consistent x-axis
    t0 = min(ep_ts[lbl][0] for lbl in active_topics)

    n_topics = len(active_topics)
    fig, axes = plt.subplots(n_topics, 1, figsize=(12, 1.6 * n_topics + 0.8),
                             sharex=True, squeeze=False)
    fig.suptitle(f"Episode {ep_idx} — inter-message interval vs. time",
                 fontsize=13, fontweight="bold", y=1.01)

    for ax_row, label in enumerate(active_topics):
        ax = axes[ax_row, 0]
        ts = ep_ts[label]
        elapsed = ts - t0
        intervals_ms = np.diff(ts) * 1000
        mid_times = elapsed[:-1] + np.diff(elapsed) / 2

        ok = intervals_ms <= actual_tolerance_ms
        n_viol = int((~ok).sum())
        ax.scatter(mid_times[ok], intervals_ms[ok], s=8, alpha=0.55,
                   color=_CLR_OK, edgecolors="none",
                   label="OK", zorder=3)
        ax.scatter(mid_times[~ok], intervals_ms[~ok], s=28, alpha=0.9,
                   color=_CLR_VIOL, edgecolors="none",
                   marker="X", label=f">{actual_tolerance_ms:.0f} ms  (n={n_viol})", zorder=4)
        ax.axhline(actual_tolerance_ms, ls="--", lw=1.2, color=_CLR_THRESH, alpha=0.5)

        # Keyframe markers (green vertical lines)
        if label in ep_kf and len(ep_kf[label]) > 0:
            kf_elapsed = ep_kf[label] - t0
            for k_i, kf_t in enumerate(kf_elapsed):
                ax.axvline(kf_t, ls="-", lw=0.8, color=PALETTE[2], alpha=0.45,
                           label="Keyframe" if k_i == 0 else None)

        ax.set_ylabel(label, fontsize=9)
        if ax_row == 0:
            ax.legend(loc="upper right", fontsize=7)

    axes[-1, 0].set_xlabel("Elapsed time (s)")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"spike_timeline_ep{ep_idx}_{DATASET_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
# ── 5b. Violation position heatmap ─────────────────────────────────────
# Rows = episodes (only those with ≥1 violation), columns = time bins.
# Left: normalised (0–100 %), right: absolute (seconds).
# Green markers on top show where AV1 keyframes cluster.

N_TIME_BINS = 50  # number of time bins along x-axis

# Collect per-episode, per-bin violation counts (normalised + absolute)
viol_ep_indices: list[int] = []  # episode indices that have violations
viol_heatmap_pct: list[np.ndarray] = []
viol_heatmap_sec: list[np.ndarray] = []

bins_pct = np.linspace(0, 100, N_TIME_BINS + 1)

# First pass: find max duration across episodes (for absolute bins)
max_duration = 0.0
for ep_idx, ep_ts in episode_timestamps.items():
    for label in topic_order_fwd:
        ts = ep_ts.get(label)
        if ts is not None and len(ts) >= 2:
            max_duration = max(max_duration, ts[-1] - ts[0])
if max_duration <= 0:
    max_duration = 10.0
bins_sec = np.linspace(0, max_duration * 1.02, N_TIME_BINS + 1)

# Keyframe positions for rug overlay
keyframe_positions_pct: dict[str, list[float]] = defaultdict(list)
keyframe_positions_sec: dict[str, list[float]] = defaultdict(list)

for ep_idx in sorted(episode_timestamps.keys()):
    ep_ts = episode_timestamps[ep_idx]
    ep_kf = episode_keyframes.get(ep_idx, {})
    row_pct = np.zeros(N_TIME_BINS)
    row_sec = np.zeros(N_TIME_BINS)
    has_viol = False

    for label in topic_order_fwd:
        ts = ep_ts.get(label)
        if ts is None or len(ts) < 2:
            continue
        intervals_ms = np.diff(ts) * 1000
        duration = ts[-1] - ts[0]
        if duration <= 0:
            continue
        mid_times = ts[:-1] + np.diff(ts) / 2
        elapsed = mid_times - ts[0]
        mask = intervals_ms > actual_tolerance_ms
        if mask.any():
            has_viol = True
            # Normalised positions (%)
            pos_pct = elapsed[mask] / duration * 100
            counts_pct, _ = np.histogram(pos_pct, bins=bins_pct)
            row_pct += counts_pct
            # Absolute positions (seconds)
            pos_sec = elapsed[mask]
            counts_sec, _ = np.histogram(pos_sec, bins=bins_sec)
            row_sec += counts_sec

        # Keyframe positions
        if label in ep_kf and len(ep_kf[label]) > 0:
            kf_elapsed = ep_kf[label] - ts[0]
            keyframe_positions_pct[label].extend((kf_elapsed / duration * 100).tolist())
            keyframe_positions_sec[label].extend(kf_elapsed.tolist())

    if has_viol:
        viol_ep_indices.append(ep_idx)
        viol_heatmap_pct.append(row_pct)
        viol_heatmap_sec.append(row_sec)

# Aggregate keyframe positions
all_kf_pct = [v for vals in keyframe_positions_pct.values() for v in vals]
all_kf_sec = [v for vals in keyframe_positions_sec.values() for v in vals]

if viol_ep_indices:
    hm_pos_pct = np.array(viol_heatmap_pct)
    hm_pos_sec = np.array(viol_heatmap_sec)
    n_viol_eps = len(viol_ep_indices)

    fig, (ax_pct, ax_sec) = plt.subplots(1, 2,
        figsize=(14, max(3, n_viol_eps * 0.18 + 1.5)))
    fig.suptitle(f"Where in the episode do violations occur? — {DATASET_TITLE}",
                 fontsize=13, fontweight="bold", y=1.02)

    for _ax in (ax_pct, ax_sec):
        _ax.grid(False)

    # ── Left: normalised (%) ──────────────────────────────────────────
    vmax_pct = float(np.max(hm_pos_pct)) if np.any(hm_pos_pct > 0) else 1
    extent_pct = [0, 100, n_viol_eps - 0.5, -0.5]
    im_pct = ax_pct.imshow(hm_pos_pct, aspect="auto", cmap="magma",
                           vmin=0, vmax=vmax_pct, extent=extent_pct,
                           interpolation="nearest")
    ax_pct.set_yticks(range(n_viol_eps))
    ax_pct.set_yticklabels([str(i) for i in viol_ep_indices], fontsize=7)
    ax_pct.set_ylabel("Episode")
    ax_pct.set_xlabel("Position within episode (%)")
    ax_pct.set_title("Normalised (%)", fontsize=10)
    fig.colorbar(im_pct, ax=ax_pct, shrink=0.7, pad=0.02, label="Violations")

    # Keyframe rug on top
    if all_kf_pct:
        ax_pct.scatter(all_kf_pct, [-0.8] * len(all_kf_pct),
                       marker="|", s=30, lw=0.6, color=PALETTE[2], alpha=0.5,
                       clip_on=False, label="Keyframes")
        ax_pct.legend(fontsize=7, loc="upper right")

    # ── Right: absolute (seconds) ────────────────────────────────────
    vmax_sec = float(np.max(hm_pos_sec)) if np.any(hm_pos_sec > 0) else 1
    extent_sec = [0, bins_sec[-1], n_viol_eps - 0.5, -0.5]
    im_sec = ax_sec.imshow(hm_pos_sec, aspect="auto", cmap="magma",
                           vmin=0, vmax=vmax_sec, extent=extent_sec,
                           interpolation="nearest")
    ax_sec.set_yticks(range(n_viol_eps))
    ax_sec.set_yticklabels([str(i) for i in viol_ep_indices], fontsize=7)
    ax_sec.set_ylabel("Episode")
    ax_sec.set_xlabel("Position within episode (s)")
    ax_sec.set_title("Absolute (seconds)", fontsize=10)
    fig.colorbar(im_sec, ax=ax_sec, shrink=0.7, pad=0.02, label="Violations")

    if all_kf_sec:
        ax_sec.scatter(all_kf_sec, [-0.8] * len(all_kf_sec),
                       marker="|", s=30, lw=0.6, color=PALETTE[2], alpha=0.5,
                       clip_on=False)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"violation_position_heatmap_{DATASET_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("✅ No tolerance violations found — nothing to plot.")

In [ ]:
# ── 5c. Episode × Topic violation heatmap ─────────────────────────────
# Rows = episodes, columns = topics.
# Cell colour = fraction of intervals exceeding tolerance_ms.
# Bright row → globally bad episode.  Bright column → chronic topic issue.

n_episodes = len(episode_timestamps)
heatmap_labels = topic_order_fwd
heatmap = np.zeros((n_episodes, len(heatmap_labels)))

for ep_idx in range(n_episodes):
    ep_ts = episode_timestamps.get(ep_idx, {})
    for col_idx, label in enumerate(heatmap_labels):
        ts = ep_ts.get(label)
        if ts is None or len(ts) < 2:
            continue
        intervals_ms = np.diff(ts) * 1000
        frac = np.mean(intervals_ms > actual_tolerance_ms)
        heatmap[ep_idx, col_idx] = frac

# Decide sizing: limit displayed episodes to keep the plot readable
MAX_SHOW = 80
if n_episodes > MAX_SHOW:
    # Show worst episodes only
    row_totals = heatmap.sum(axis=1)
    show_idx = np.argsort(row_totals)[-MAX_SHOW:]
    show_idx.sort()
    heatmap_show = heatmap[show_idx]
    y_labels = [str(i) for i in show_idx]
    subtitle = f"  (showing {MAX_SHOW} worst of {n_episodes} episodes)"
else:
    heatmap_show = heatmap
    y_labels = [str(i) for i in range(n_episodes)]
    subtitle = ""

fig, ax = plt.subplots(figsize=(max(6, len(heatmap_labels) * 0.9),
                                max(4, len(y_labels) * 0.22 + 1)))
ax.grid(False)
hm_vmax_5c = float(np.max(heatmap_show)) if np.any(heatmap_show > 0) else 0.05
im = ax.imshow(heatmap_show, aspect="auto", cmap="magma", vmin=0, vmax=hm_vmax_5c)

ax.set_xticks(range(len(heatmap_labels)))
ax.set_xticklabels(heatmap_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(y_labels)))
ax.set_yticklabels(y_labels, fontsize=7)
ax.set_ylabel("Episode")
ax.set_title(f"Fraction of intervals > {actual_tolerance_ms:.0f} ms{subtitle}")

cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label("Violation fraction", fontsize=9)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"violation_heatmap_{DATASET_LABEL}.png",
            dpi=200, bbox_inches="tight")
plt.show()

---
## 6. One-Pager PDF Report

Generates a single landscape-A4 PDF summarising dataset quality at a glance:

| Row | Left panel | Right panel |
|-----|-----------|-------------|
| **1** | Metadata + timestamp source table | Frequency boxplot (per-topic avg Hz) |
| **2** | Key violation statistics | Episode × topic heatmap |
| **3** | Violation position histogram (aggregate, % + seconds) with keyframe markers | ← full width |

Automated verdict: **PASS** / **WARNING** / **FAIL** based on configurable thresholds.

In [ ]:
    ("Dataset Version", dataset_version_str or "—"),